In [ ]:
import numpy as onp
from pathlib import Path
import sys
_project_root: Path = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[1]))
import src.symmetry as sym
from src.categorization import show_latex_bmatrix
from src.bases import CanonBasis4x4
import src.permutations as perms

In [ ]:
# shift op
def initKshit(p):
  # X from homework
  perm = perms.perm_from_cycles(((1,2,3),))
  X = perms.perm_hom(perm)
  # C3 = perms.generated_group((perm,))
  C3 = sym.generated_group(
    [sym.AffineOperation(X, label="(123)")],
    name="C3",
    max_order=3,
  )
  U = sym.cyclic_unitary_rep(X, 3, name="shift rep")
  print(U.matrices.values())
  print(C3.matrices)
  ps = onp.sqrt(onp.array([1-p, p/2, p/2])).astype(onp.float32)
  Kg = ps * C3.matrices
  # print(ps[:, None, None] * C3.matrices)
  return Kg
  # K0 = onp.sqrt(1 - p) * I
  # K1 = onp.sqrt(p / 2) * g123
  # K2 = onp.sqrt(p / 2) * (g123 @ g123)
  # return (K0, K1, K2)

# clock op
def initKcock2(p):
  ω = onp.exp(2j * onp.pi / 3)
  Z = onp.diag([1, ω, ω**2])

  I = onp.eye(3, dtype=complex)
  K0 = onp.sqrt(1 - p) * I
  K1 = onp.sqrt(p) * Z
  return (K0, K1)

def init_clock3():
  ω = onp.exp(2j * onp.pi / 3)
  Z = onp.diag([1, ω, ω**2])
  U = cyclic_unitary_rep(Z, 3, name="clock rep")
  print(U)
  # C_3 cyclic group of "clock" matrices
  return sym.generated_group(
    [sym.AffineOperation(Z, label="")],
    name="C3",
    max_order=3,
  )

def channel(ψ: onp.ndarray, OSR: tuple[onp.ndarray, ...]|onp.ndarray):
  ρ = onp.outer(ψ, ψ.conj())
  ρ1 = onp.array(sum(K @ ρ @ K.conj().T for K in OSR))
  ρI = onp.array(sum(K.conj().T @ K for K in OSR)) # completenes
  return ρ1, ρI


p = 2 / 3
shift_OSR = initKshit(p)
shift_ψ = onp.array([1, 0, 0], dtype=complex)
rho_shift, comp_shift = channel(shift_ψ, shift_OSR)

# print("Shift channel rho' at p=2/3:")
# print(onp.round(rho_shift, 4))

# print("\nShift channel completeness:")
# print(onp.round(comp_shift, 4))

p = 0.5
clock_OSR = initKcock2(p)
clock_ψ = onp.ones(3, dtype=complex) / onp.sqrt(3)
rho_clock, comp_clock = channel(clock_ψ, clock_OSR)

# print("\nClock channel rho' at p=1/2:")
# print(onp.round(rho_clock, 4))

# print("\nClock channel completeness:")
# print(onp.round(comp_clock, 4))

# print("\nOff-diagonal magnitude at p=1/2:")
# print(abs(rho_clock[0, 1]))

dict_values([array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]]), array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]]), array([[0., 1., 0.],
       [0., 0., 1.],
       [1., 0., 0.]])])
[[[1. 0. 0.]
  [0. 1. 0.]
  [0. 0. 1.]]

 [[0. 0. 1.]
  [1. 0. 0.]
  [0. 1. 0.]]

 [[0. 1. 0.]
  [0. 0. 1.]
  [1. 0. 0.]]]


In [1]:
import sympy as sp
a = sp.symbols('a', rational=True)
x = sp.symbols('x')

root = sp.sqrt(2) + a
display(sp.expand(root ** 3))
display(sp.expand((3 * a ** 2 + 2) * root))

f = x ** 3 - (3 * a ** 2 + 2) * x + 2 * a ** 3 - 4 * a
display(sp.factor(f))
sp.roots(f, x)



a**3 + 3*sqrt(2)*a**2 + 6*a + 2*sqrt(2)

3*a**3 + 3*sqrt(2)*a**2 + 2*a + 2*sqrt(2)

(2*a + x)*(a**2 - 2*a*x + x**2 - 2)

{-2*a: 1, a - sqrt(2): 1, a + sqrt(2): 1}

In [ ]:
import sympy as sp

a, b, c, d = sp.symbols("a b c d")
p, γ = sp.symbols("p γ", real=True, nonnegative=True)

ρ = onp.array([[a, b], [c, d]], dtype=object)

s1 = sp.sqrt(p)
s2 = sp.sqrt(γ)
s3 = sp.sqrt(1 - γ)


B = CanonBasis4x4()
E11, E12, E21, E22 = map(lambda x: x.astype(object), B)


E0 = sp.sqrt(p) * (E11 + sp.sqrt(1 - γ) * E22)
E1 = sp.sqrt(p) * (sp.sqrt(γ) * E12)
E2 = sp.sqrt(1 - p) * (sp.sqrt(1 - γ) * E11 + E22)
E3 = sp.sqrt(1 - p) * (sp.sqrt(γ) * E21)

Ks = [E0, E1, E2, E3]

out = onp.zeros(ρ.shape, dtype=object)
for K in Ks:
  out += K @ ρ @ K.T#.conjugate() # all terms reall

# def conj(x: sp.core.add.Add):
#   return sp.simplify(x.xreplace({
#     sp.conjugate(s1): s1,
#     sp.conjugate(s2): s2,
#     sp.conjugate(s3): s3,
#   }))

res = onp.vectorize(sp.simplify)(out)
show_latex_bmatrix(res, name=r"\Phi(\rho)")


<IPython.core.display.Math object>

In [ ]:
import qutip as qt


def z3_clock_operator():
  omega = onp.exp(2j * onp.pi / 3)
  return qt.Qobj(
    onp.diag([1, omega, omega**2]),
    dims=[[3], [3]],
  )


def z3_clock_osr(ps):
  ps = onp.asarray(ps, dtype=float)
  if ps.shape != (3,):
    raise ValueError("probabilities must be a length-3 array")
  if onp.any(ps < 0) or not onp.isclose(ps.sum(), 1.0):
    raise ValueError("probabilities must be nonnegative and sum to 1")

  Z = z3_clock_operator()
  powers = [qt.qeye(3), Z, Z**2]
  return [onp.sqrt(pk) * U for pk, U in zip(ps, powers)]


def z3_clock_osr_from_p(p):
  if not 0.0 <= p <= 1.0:
    raise ValueError("p must lie in [0, 1]")
  return z3_clock_osr([1.0 - p, p / 2.0, p / 2.0])


def apply_osr(rho, kraus_ops):
  if rho.isket:
    rho = qt.ket2dm(rho)
  if rho.shape != (3, 3):
    raise ValueError("rho must be a qutrit ket or a 3x3 density matrix")

  rho_out = 0 * rho
  for K in kraus_ops:
    rho_out += K * rho * K.dag()
  return rho_out


def kraus_completeness(kraus_ops):
  comp = 0 * kraus_ops[0]
  for K in kraus_ops:
    comp += K.dag() * K
  return comp


p: float = 0.5
clock_osr = z3_clock_osr([1/3,1/3,1/3])
# clock_osr = z3_clock_osr_from_p(p)

psi_a = (qt.basis(3, 0) + onp.exp(1j * onp.pi / 4) * qt.basis(3, 1)).unit()
psi_b = (qt.basis(3, 1) + 2 * qt.basis(3, 2)).unit()
rho = 0.65 * qt.ket2dm(psi_a) + 0.35 * qt.ket2dm(psi_b)

rho_out = apply_osr(rho, clock_osr)
comp = kraus_completeness(clock_osr)
clock_super = qt.kraus_to_super(clock_osr)

print("Kraus operators K_k for the Z/3Z clock channel:")
for k, K in enumerate(clock_osr):
  print(f"\nK_{k} =")
  print(K.full())

print("\nInput rho:")
show_latex_bmatrix(rho.full())

print("\nOutput E(rho):")
show_latex_bmatrix(onp.abs(rho_out.full()))

print("\nCompleteness sum_k K_k^dagger K_k:")
show_latex_bmatrix(comp.full())

print("\nSuperoperator dims:")
print(clock_super.dims)


Kraus operators K_k for the Z/3Z clock channel:

K_0 =
[[0.58+0.j 0.  +0.j 0.  +0.j]
 [0.  +0.j 0.58+0.j 0.  +0.j]
 [0.  +0.j 0.  +0.j 0.58+0.j]]

K_1 =
[[ 0.58+0.j   0.  +0.j   0.  +0.j ]
 [ 0.  +0.j  -0.29+0.5j  0.  +0.j ]
 [ 0.  +0.j   0.  +0.j  -0.29-0.5j]]

K_2 =
[[ 0.58+0.j   0.  +0.j   0.  +0.j ]
 [ 0.  +0.j  -0.29-0.5j  0.  +0.j ]
 [ 0.  +0.j   0.  +0.j  -0.29+0.5j]]

Input rho:


<IPython.core.display.Math object>


Output E(rho):


<IPython.core.display.Math object>


Completeness sum_k K_k^dagger K_k:


<IPython.core.display.Math object>


Superoperator dims:
[[[3], [3]], [[3], [3]]]
